# AI Technical Solutions Advisor
### Ed Donner LLM Engineering · Week 1 exercise

**Original requirement:** Take a technical question and respond with an explanation using the **OpenAI API** and **Ollama**.

**How it works**
1. Enter any technical question (or paste code you want explained).
2. Ask **GPT-4o-mini** through OpenAI, streaming the answer as it arrives.
3. Ask **Llama 3.2** through local Ollama, also streaming the answer.
4. See a quick side-by-side comparison of basic performance metrics.

Both models receive **the same question and system instructions**, so you can compare how they teach a topic.

In [ ]:
import os
import time
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from ollama import Client as OllamaClient

## 2 · Constants

Here are two example models. You can change them if you have access to other models.

In [ ]:
MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434")

# Turn either provider off if you only want to run one model.
RUN_OPENAI = True
RUN_OLLAMA = True

## 3 · Set up environment

In [ ]:
# Search the current directory and its parents, which usually finds the course's root .env.
env_file = find_dotenv(usecwd=True)
if env_file:
    load_dotenv(env_file)

api_key = os.getenv("OPENAI_API_KEY", "").strip()
print("OpenAI key configured:", "Yes" if api_key else "No — add OPENAI_API_KEY to .env")
print("Ollama endpoint:", OLLAMA_HOST)
print("OpenAI enabled:", RUN_OPENAI, "| Ollama enabled:", RUN_OLLAMA)

## 4 · Ask a technical question

Edit just the string below, then re-run this cell and the model-answer cells.

**Example questions**
AI, LLMs, and machine learning
- What is the difference between an LLM, a foundation model, and a traditional machine learning model?
- Explain how transformers and the attention mechanism work.
- What is the difference between tokens, embeddings, and vectors?
- How does Retrieval-Augmented Generation (RAG) reduce hallucinations?
- What is the difference between prompt engineering, fine-tuning, and RAG?
- Explain temperature, top-p, and context windows in large language models.
- What is quantization, and why is it important for running models locally?
- How do AI agents differ from traditional chatbots?
- What is the difference between LoRA and QLoRA?
- How would you evaluate the accuracy and reliability of an LLM?

In [ ]:

question = """
What is the difference between an LLM, a foundation model,
and a traditional machine learning model?

Explain each concept, provide practical examples,
and describe when each approach should be used.
""".strip()

assert question, "Please enter a technical question first."

display(Markdown(f"**Your technical question:**\n\n{question}"))


## 5 · Shared system prompt

In [ ]:
SYSTEM_PROMPT = """You are an accurate, approachable technical tutor helping someone learn
Python, APIs, software engineering, databases, AI and enterprise architecture.

Answer the user's technical question in Markdown. Include these sections:
## Short answer
## Step-by-step explanation
## Practical example
## Common mistakes or limitations
## Key takeaway

Use clear, beginner-friendly language while remaining technically correct.
When the question contains code, explain important lines and provide a minimal example
when useful. Do not execute code supplied by the user. If the code needs surrounding
context to run, say so. Explain assumptions and uncertainty instead of inventing facts.
Keep the answer focused and useful, normally under 600 words.
"""


def make_messages(technical_question):
    if not isinstance(technical_question, str) or not technical_question.strip():
        raise ValueError("Enter a non-empty technical question.")
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": technical_question.strip()},
    ]

## 6 · Streaming output helper

In [ ]:
def display_stream(chunks):
    """Render a stream of text fragments as Markdown and return the full answer."""
    handle = display(Markdown("_Generating explanation…_"), display_id=True)
    fragments = []
    last_refresh = time.monotonic()

    for fragment in chunks:
        if not fragment:
            continue
        fragments.append(fragment)
        now = time.monotonic()
        if now - last_refresh >= 0.25:
            handle.update(Markdown("".join(fragments) + " ▌"))
            last_refresh = now

    answer = "".join(fragments)
    handle.update(Markdown(answer if answer else "_The model returned no text._"))
    return answer

## 7 · OpenAI: GPT-4o-mini (streaming)

This is a **real OpenAI API call**, using the Python SDK's streamed Chat Completions. It needs a valid API key and an internet connection. OpenAI API calls can incur charges.

In [ ]:
def explain_with_openai(technical_question):
    """Ask GPT-4o-mini and return its explanation plus simple timing metrics."""
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key:
        raise RuntimeError("Missing OPENAI_API_KEY. Add it to your .env file and rerun setup.")

    client = OpenAI(api_key=key, timeout=90.0)
    messages = make_messages(technical_question)
    started = time.perf_counter()

    events = client.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        stream=True,
    )

    def text_fragments():
        for event in events:
            if event.choices:
                yield event.choices[0].delta.content or ""

    answer = display_stream(text_fragments())
    return {"provider": "OpenAI", "model": MODEL_GPT, "answer": answer,
            "seconds": time.perf_counter() - started}


openai_result = None
if not RUN_OPENAI:
    print("OpenAI skipped (RUN_OPENAI=False).")
else:
    try:
        display(Markdown("### OpenAI explanation"))
        openai_result = explain_with_openai(question)
        print(f"OpenAI completed in {openai_result['seconds']:.2f} seconds.")
    except Exception as exc:
        print(f"OpenAI unavailable: {type(exc).__name__}: {exc}")
        print("Check the API key, network, account billing and model access.")

## 8 · Local Ollama: Llama 3.2 (streaming)

This makes a **real request to Ollama running on your computer**. Ollama does not require your OpenAI key. Start Ollama and download the model with `ollama pull llama3.2` before running this cell.

In [ ]:
def explain_with_ollama(technical_question):
    """Ask local Llama 3.2 and return its explanation plus simple timing metrics."""
    client = OllamaClient(host=OLLAMA_HOST, timeout=180.0)
    messages = make_messages(technical_question)
    started = time.perf_counter()

    events = client.chat(model=MODEL_LLAMA, messages=messages, stream=True)

    def text_fragments():
        for event in events:
            yield event.message.content or ""

    answer = display_stream(text_fragments())
    return {"provider": "Ollama", "model": MODEL_LLAMA, "answer": answer,
            "seconds": time.perf_counter() - started}


ollama_result = None
if not RUN_OLLAMA:
    print("Ollama skipped (RUN_OLLAMA=False).")
else:
    try:
        display(Markdown("### Ollama explanation"))
        ollama_result = explain_with_ollama(question)
        print(f"Ollama completed in {ollama_result['seconds']:.2f} seconds.")
    except Exception as exc:
        print(f"Ollama unavailable: {type(exc).__name__}: {exc}")
        print("Check that Ollama is running, then run: ollama pull llama3.2")

## 9 · Compare the models

Compare **elapsed time and answer length**.

In [ ]:
results = [r for r in (openai_result, ollama_result) if r is not None]
if results:
    rows = ["| Provider | Model | Time (seconds) | Characters |",
            "|---|---|---:|---:|"]
    for result in results:
        rows.append(
            f"| {result['provider']} | `{result['model']}` | "
            f"{result['seconds']:.2f} | {len(result['answer']):,} |"
        )
    display(Markdown("\n".join(rows)))
    display(Markdown("**Reflection:** Which explanation was clearer, more accurate, and more useful? "
                     "What differences did you notice?"))
else:
    print("No model responses available. Run either model cell after setup.")